# XQuality — Evaluate completed prefix stop-after-layer outputs only

This notebook **does not call OpenRouter**. It evaluates the already generated prefix stop-after-layer outputs.

It automatically excludes incomplete folders. In particular, if you stopped during `prefix_stop_after_06_*` and this is the last available prefix without a completion marker, it is excluded by default.

Expected source folder:

```text
examples/XQualityMachine32/runs/prefix_stop_after_layer_llm_finalization_exact_eval/
```

Outputs:

```text
examples/XQualityMachine32/runs/prefix_stop_after_layer_llm_finalization_exact_eval/evaluation_completed_only/
```


In [ ]:

from __future__ import annotations

import json
import re
import sys
from pathlib import Path
from typing import Any
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 140)


In [ ]:

# -------------------------
# Repository / path config
# -------------------------

def find_repo_root(start: Path | None = None) -> Path:
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / 'src' / 'neoolaf').exists() or (p / 'pyproject.toml').exists():
            return p
    # fallback for notebooks launched from examples/XQualityMachine32
    for p in candidates:
        if (p / 'examples' / 'XQualityMachine32').exists():
            return p
    raise RuntimeError(f'Could not find NeoOLAF repo root from {start}')

ROOT = find_repo_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PROFILE_NAME = 'xquality_loose'

GOLD_EXCEL = ROOT / 'data' / 'XQuality' / 'Machine32' / 'machine32_triples.xlsx'
GOLD_JSON = ROOT / 'data' / 'XQuality' / 'Examples' / 'XQuality_all_triplets_flat_en.json'
GOLD_PATH = GOLD_EXCEL if GOLD_EXCEL.exists() else GOLD_JSON

PREFIX_OUTPUT_ROOT = ROOT / 'examples' / 'XQualityMachine32' / 'runs' / 'prefix_stop_after_layer_llm_finalization_exact_eval'
EVAL_OUTPUT_DIR = PREFIX_OUTPUT_ROOT / 'evaluation_completed_only'
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AUTO_EXCLUDE_INCOMPLETE = True
AUTO_EXCLUDE_LAST_UNMARKED = True
# If you are sure layer 6 is complete even without a marker, set this empty.
FORCE_EXCLUDE_STOP_INDEXES_IF_INCOMPLETE = {6}

print('ROOT =', ROOT)
print('GOLD_PATH =', GOLD_PATH, '| exists:', GOLD_PATH.exists())
print('PREFIX_OUTPUT_ROOT =', PREFIX_OUTPUT_ROOT, '| exists:', PREFIX_OUTPUT_ROOT.exists())
print('EVAL_OUTPUT_DIR =', EVAL_OUTPUT_DIR)


In [ ]:

# -------------------------
# Import exact NeoOLAF evaluator stack
# -------------------------
from neoolaf.evaluation.runners.evaluate_layer_state import load_xquality_gold_any, gold_to_artifact
from neoolaf.evaluation.metrics.extraction import evaluate_extraction
from neoolaf.evaluation.profiles.registry import get_profile
from neoolaf.evaluation.schema.artifact import EvalDocument, EvalEntity, EvalRelation, EvaluationArtifact

profile = get_profile(PROFILE_NAME)
gold = load_xquality_gold_any(GOLD_PATH)
gold_artifact = gold_to_artifact(gold, profile=PROFILE_NAME)
print('Loaded gold with exact NeoOLAF loader.')
print('Gold entities:', sum(len(v) for v in gold_artifact.entities_by_doc.values()))
print('Gold relations:', sum(len(v) for v in gold_artifact.relations_by_doc.values()))


In [ ]:

# -------------------------
# Helpers: discover and validate prefix folders
# -------------------------
LAYER_RE = re.compile(r'prefix_stop_after_(\d+)_')

def get_stop_index(path: Path) -> int | None:
    m = LAYER_RE.search(path.name)
    return int(m.group(1)) if m else None


def load_json_safe(path: Path, default=None):
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        return default


def triples_path_for(folder: Path) -> Path:
    for name in ['triples.json', 'llm_triples.json']:
        p = folder / name
        if p.exists():
            return p
    return folder / 'triples.json'


def inspect_prefix_folder(folder: Path, max_index_found: int | None = None) -> dict[str, Any]:
    idx = get_stop_index(folder)
    triples_path = triples_path_for(folder)
    completed_marker = folder / 'COMPLETED.json'
    success_marker = folder / '_SUCCESS.json'
    metadata_path = folder / 'metadata.json'
    batch_meta = folder / 'batch_metadata.csv'

    triples = load_json_safe(triples_path, default=None) if triples_path.exists() else None
    triples_valid = isinstance(triples, list)
    triple_count = len(triples) if triples_valid else 0

    marker_exists = completed_marker.exists() or success_marker.exists()
    marker = load_json_safe(completed_marker, default={}) if completed_marker.exists() else load_json_safe(success_marker, default={}) if success_marker.exists() else {}

    batch_rows = None
    batch_errors = None
    if batch_meta.exists():
        try:
            bdf = pd.read_csv(batch_meta)
            batch_rows = len(bdf)
            # Be flexible with possible column names.
            error_cols = [c for c in bdf.columns if c.lower() in {'error', 'parse_error', 'exception'}]
            if error_cols:
                batch_errors = int(sum(bdf[c].notna().sum() for c in error_cols))
            elif 'status' in bdf.columns:
                batch_errors = int((bdf['status'].astype(str).str.lower().isin(['error', 'failed', 'fail'])).sum())
        except Exception:
            batch_rows = None

    reason = []
    complete = True
    if not triples_path.exists():
        complete = False
        reason.append('missing_triples_json')
    elif not triples_valid:
        complete = False
        reason.append('invalid_triples_json')
    elif triple_count == 0:
        complete = False
        reason.append('empty_triples_json')

    # If the last existing folder has no completion marker, it is probably the interrupted stop point.
    if AUTO_EXCLUDE_LAST_UNMARKED and idx is not None and max_index_found is not None:
        if idx == max_index_found and not marker_exists:
            complete = False
            reason.append('last_prefix_without_completion_marker')

    if idx in FORCE_EXCLUDE_STOP_INDEXES_IF_INCOMPLETE and not marker_exists:
        # This only excludes it if it already looks suspicious through lack of marker.
        # Useful for your interrupted layer 6 run.
        if idx == max_index_found or triple_count == 0:
            complete = False
            reason.append(f'forced_exclude_stop_{idx}_without_marker')

    return {
        'folder': str(folder),
        'folder_name': folder.name,
        'stop_index': idx,
        'triples_path': str(triples_path),
        'triples_exists': triples_path.exists(),
        'triple_count': triple_count,
        'completion_marker_exists': marker_exists,
        'batch_metadata_exists': batch_meta.exists(),
        'batch_rows': batch_rows,
        'batch_errors': batch_errors,
        'complete': complete,
        'exclude_reason': ';'.join(reason),
    }

folders = sorted([p for p in PREFIX_OUTPUT_ROOT.glob('prefix_stop_after_*') if p.is_dir()], key=lambda p: (get_stop_index(p) if get_stop_index(p) is not None else 999, p.name))
max_idx = max([get_stop_index(p) for p in folders if get_stop_index(p) is not None], default=None)
inspection = pd.DataFrame([inspect_prefix_folder(p, max_idx) for p in folders])
inspection.to_csv(EVAL_OUTPUT_DIR / 'prefix_folder_inspection.csv', index=False)
display(inspection)

completed_df = inspection[inspection['complete']].copy()
excluded_df = inspection[~inspection['complete']].copy()
completed_df.to_csv(EVAL_OUTPUT_DIR / 'included_prefixes.csv', index=False)
excluded_df.to_csv(EVAL_OUTPUT_DIR / 'excluded_prefixes.csv', index=False)

print('Included prefixes:', len(completed_df))
print('Excluded prefixes:', len(excluded_df))
if len(excluded_df):
    display(excluded_df[['stop_index', 'folder_name', 'triple_count', 'completion_marker_exists', 'exclude_reason']])


In [ ]:

# -------------------------
# Convert generated triples to exact NeoOLAF EvaluationArtifact
# -------------------------

def clean_label(x: Any) -> str:
    if x is None:
        return ''
    return re.sub(r'\s+', ' ', str(x).strip())


def canonical_predicate(x: Any) -> str:
    s = clean_label(x).upper()
    s = re.sub(r'[^A-Z0-9]+', '_', s).strip('_')
    aliases = {
        'TRIGGER': 'TRIGGERS',
        'TRIGGERS': 'TRIGGERS',
        'CAUSE': 'CAUSES',
        'CAUSES': 'CAUSES',
        'REQUIRE': 'REQUIRES',
        'REQUIRES': 'REQUIRES',
        'HANDLED_BY': 'HANDLED_BY',
        'HANDLES': 'HANDLED_BY',
        'RESPONSIBLE': 'HANDLED_BY',
        'REFERENCE': 'REFERENCES',
        'REFERENCES': 'REFERENCES',
    }
    return aliases.get(s, s)


def normalize_triples(raw: Any) -> list[dict[str, Any]]:
    if isinstance(raw, dict) and 'triples' in raw:
        raw = raw['triples']
    if not isinstance(raw, list):
        return []
    out = []
    seen = set()
    for i, t in enumerate(raw):
        if not isinstance(t, dict):
            continue
        subj = clean_label(t.get('subject_label') or t.get('subject') or t.get('head') or t.get('source'))
        pred = canonical_predicate(t.get('predicate') or t.get('relation') or t.get('property'))
        obj = clean_label(t.get('object_label') or t.get('object') or t.get('tail') or t.get('target'))
        if not subj or not pred or not obj:
            continue
        key = (subj.lower(), pred, obj.lower())
        if key in seen:
            continue
        seen.add(key)
        out.append({
            'triple_id': t.get('triple_id') or f'pred_{len(out):05d}',
            'subject_label': subj,
            'predicate': pred,
            'object_label': obj,
            'confidence': t.get('confidence'),
            'chunk_id': t.get('chunk_id') or t.get('unit_id') or t.get('source_unit_id'),
            'raw': t,
        })
    return out


def artifact_from_generated_triples(triples: list[dict[str, Any]], *, run_id: str, profile_name: str, source_folder: str) -> EvaluationArtifact:
    doc_id = 'xquality'
    artifact = EvaluationArtifact(
        method='prefix_stop_after_layer_llm_finalization',
        dataset='xquality',
        profile=profile_name,
        run_id=run_id,
        metadata={'source_folder': source_folder, 'evaluation_input': 'generated_triples_json'},
    )
    artifact.documents.append(EvalDocument(document_id=doc_id, metadata={'dataset': 'xquality'}))

    entities: dict[str, EvalEntity] = {}
    relations: list[EvalRelation] = []

    def add_entity(label: str):
        label = clean_label(label)
        if not label:
            return
        key = label.lower()
        if key not in entities:
            entities[key] = EvalEntity(label=label, type='generated_triple_endpoint', provenance_present=False, raw={})

    for t in triples:
        h = clean_label(t.get('subject_label'))
        r = canonical_predicate(t.get('predicate'))
        o = clean_label(t.get('object_label'))
        if not (h and r and o):
            continue
        add_entity(h)
        add_entity(o)
        relations.append(EvalRelation(
            head=h,
            relation=r,
            tail=o,
            evidence='',
            confidence=t.get('confidence'),
            provenance_present=bool(t.get('chunk_id')),
            provenance={'chunk_id': t.get('chunk_id') or ''},
            raw=t.get('raw') or t,
        ))

    artifact.entities_by_doc[doc_id] = sorted(entities.values(), key=lambda e: e.label.lower())
    artifact.relations_by_doc[doc_id] = relations
    return artifact


def flatten_metrics(extraction: dict[str, Any]) -> dict[str, Any]:
    # Support a few possible evaluator output shapes.
    ent = extraction.get('entity') or extraction.get('entities') or extraction.get('entity_metrics') or {}
    rel = extraction.get('relation') or extraction.get('relations') or extraction.get('relation_metrics') or {}
    counts = extraction.get('counts', {}) or {}
    return {
        'entity_tp': ent.get('tp'),
        'entity_fp': ent.get('fp'),
        'entity_fn': ent.get('fn'),
        'entity_precision': ent.get('precision'),
        'entity_recall': ent.get('recall'),
        'entity_f1': ent.get('f1'),
        'relation_tp': rel.get('tp'),
        'relation_fp': rel.get('fp'),
        'relation_fn': rel.get('fn'),
        'relation_precision': rel.get('precision'),
        'relation_recall': rel.get('recall'),
        'relation_f1': rel.get('f1'),
        'counts': counts,
    }


def extract_per_relation_rows(extraction: dict[str, Any], base: dict[str, Any]) -> list[dict[str, Any]]:
    per = extraction.get('per_relation') or extraction.get('relations_by_type') or {}
    rows = []
    if isinstance(per, dict):
        for relation, m in per.items():
            if not isinstance(m, dict):
                continue
            row = dict(base)
            row.update({
                'predicate': relation,
                'tp': m.get('tp'),
                'fp': m.get('fp'),
                'fn': m.get('fn'),
                'precision': m.get('precision'),
                'recall': m.get('recall'),
                'f1': m.get('f1'),
                'pred_count': m.get('pred_count'),
                'gold_count': m.get('gold_count'),
            })
            rows.append(row)
    return rows


In [ ]:

# -------------------------
# Evaluate completed prefixes
# -------------------------
summary_rows = []
per_relation_rows = []
raw_results = []

for _, row in tqdm(completed_df.iterrows(), total=len(completed_df), desc='Evaluating completed prefixes'):
    folder = Path(row['folder'])
    triples_path = Path(row['triples_path'])
    raw = load_json_safe(triples_path, default=[])
    triples = normalize_triples(raw)

    run_id = folder.name
    artifact = artifact_from_generated_triples(
        triples,
        run_id=run_id,
        profile_name=PROFILE_NAME,
        source_folder=str(folder),
    )
    extraction = evaluate_extraction(artifact, gold_artifact, profile)
    flat = flatten_metrics(extraction)

    base = {
        'series': 'prefix_stop_after_layer_generated_output',
        'layer_name': folder.name,
        'stop_index': row['stop_index'],
        'evaluation_mode': 'exact_neoolaf_evaluate_extraction',
        'profile': PROFILE_NAME,
        'triple_count': len(triples),
        'source_folder': str(folder),
    }
    summary = {**base, **{k: v for k, v in flat.items() if k != 'counts'}}
    summary_rows.append(summary)
    per_relation_rows.extend(extract_per_relation_rows(extraction, base))
    raw_results.append({'base': base, 'extraction': extraction})

summary_df = pd.DataFrame(summary_rows).sort_values(['stop_index', 'layer_name'])
per_relation_df = pd.DataFrame(per_relation_rows).sort_values(['stop_index', 'predicate']) if per_relation_rows else pd.DataFrame()

summary_df.to_csv(EVAL_OUTPUT_DIR / 'summary_exact_eval_completed.csv', index=False)
per_relation_df.to_csv(EVAL_OUTPUT_DIR / 'per_relation_exact_eval_completed.csv', index=False)
(EVAL_OUTPUT_DIR / 'raw_eval_results.json').write_text(json.dumps(raw_results, indent=2, ensure_ascii=False), encoding='utf-8')

display(summary_df)
if not per_relation_df.empty:
    display(per_relation_df.head(30))
print('Saved:', EVAL_OUTPUT_DIR)


In [ ]:

# -------------------------
# Optional native final reference from saved Layer 12 state
# -------------------------
from neoolaf.evaluation.runners.evaluate_layer_state import evaluate_layer_state

ADD_NATIVE_LAYER12_REFERENCE = True
NATIVE_LAYER12_STATE = None  # set manually if auto-discovery chooses wrong path

LAYER12_CANDIDATES = []
if NATIVE_LAYER12_STATE is not None:
    LAYER12_CANDIDATES = [Path(NATIVE_LAYER12_STATE)]
else:
    search_root = ROOT / 'examples' / 'XQualityMachine32' / 'runs'
    LAYER12_CANDIDATES = sorted(
        search_root.rglob('layer12_serialization/state.json'),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

native_rows = []
if ADD_NATIVE_LAYER12_REFERENCE and LAYER12_CANDIDATES:
    state_path = LAYER12_CANDIDATES[0]
    print('Using native Layer 12 state:', state_path)
    native = evaluate_layer_state(
        state_path=state_path,
        gold_path=GOLD_PATH,
        profile_name=PROFILE_NAME,
        layer_name='native_neoolaf_layer12_state',
        output_path=EVAL_OUTPUT_DIR / 'native_layer12_state_eval.json',
    )
    flat = flatten_metrics(native.get('extraction', {}))
    native_row = {
        'series': 'native_neoolaf_layer12_state',
        'layer_name': 'native_neoolaf_layer12_state',
        'stop_index': 12,
        'evaluation_mode': 'evaluate_layer_state',
        'profile': PROFILE_NAME,
        'triple_count': native.get('counts', {}).get('pred_relations'),
        'source_folder': str(state_path.parent),
        **{k: v for k, v in flat.items() if k != 'counts'},
    }
    native_rows.append(native_row)
    combined = pd.concat([summary_df, pd.DataFrame(native_rows)], ignore_index=True)
else:
    combined = summary_df.copy()

combined.to_csv(EVAL_OUTPUT_DIR / 'summary_exact_eval_completed_with_native_reference.csv', index=False)
display(combined)


In [ ]:

# -------------------------
# Plots
# -------------------------
plot_df = combined.copy()
plot_df = plot_df[pd.notna(plot_df['relation_f1'])].sort_values(['series', 'stop_index'])

plt.figure(figsize=(11, 5))
for series, sdf in plot_df.groupby('series'):
    plt.plot(sdf['stop_index'], sdf['relation_f1'], marker='o', label=series)
plt.xlabel('Stop-after layer index')
plt.ylabel('Relation F1')
plt.title('Prefix stop-after-layer output evaluation')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(EVAL_OUTPUT_DIR / 'relation_f1_prefix_stop_after_layer.png', dpi=200)
plt.show()

plt.figure(figsize=(11, 5))
for series, sdf in plot_df.groupby('series'):
    plt.plot(sdf['stop_index'], sdf['relation_precision'], marker='o', label=f'{series} precision')
    plt.plot(sdf['stop_index'], sdf['relation_recall'], marker='x', label=f'{series} recall')
plt.xlabel('Stop-after layer index')
plt.ylabel('Score')
plt.title('Relation precision/recall by prefix stop point')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(EVAL_OUTPUT_DIR / 'relation_precision_recall_prefix_stop_after_layer.png', dpi=200)
plt.show()


## Interpretation

Use `summary_exact_eval_completed_with_native_reference.csv` for the paper/report.

The excluded prefixes are listed in `excluded_prefixes.csv`. If Layer 6 was interrupted, it should appear there unless you explicitly created a completion marker or disabled the exclusion policy.
